# Indian Household Data Cleaning Pipeline

**Project**
**Objective:** Sequentially clean, standardize, and verify CMIE datasets (Aspirations of India, Consumption Pyramids, Household Income, People of India). 

This notebook processes the raw `.csv` files through **Stage 1 to Stage 8**, applying memory-efficient schema standardization, non-response flagging, time-granularity matching, and primary key audits.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Global Configuration
base_dir = r"Household Project"
datasets = ["Aspirations_of_india", "consumption_pyramids", "household_income", "people_of_india"]

# CMIE Analysis Year - Used to derive wave labels when only month numbers are present
ANALYSIS_YEAR = 2025

## Stage 1: Standardize Schema (Columns & Data Types)

**Why is this stage here?**
Raw monthly survey files often have small inconsistencies. A column might be called `HH_ID` in January and `hh_id` in February. Similarly, a numeric column might accidentally contain a text value like `"N/A"` in one month, causing pandas to load the entire column as `object` (text) instead of `float64` or `int64`. 

Stage 1 performs a two-pass scan:
1. **Pass 1:** Scans all files to find every unique column and determines the "safest" combined data type (e.g., if one file is `int64` and another is `float64`, it promotes to `float64`).
2. **Pass 2:** Applies these consistent column names and data types, ensuring standard structures across all months.

In [ ]:
ID_COLS = {'HH_ID', 'MEM_ID'}

def strip_trailing_float(x):
    """Convert '12345.0' -> '12345' but leave '12345.5' untouched."""
    if isinstance(x, str):
        x = x.strip()
        if '.' in x:
            head, _, tail = x.partition('.')
            if tail.strip('0') == '':
                return head
    return x

def resolve_dtype(dtype_strs):
    """Resolve mismatched datatypes across files."""
    dtype_strs = set(dtype_strs)
    if any(('object' in d) or ('bool' in d) or ('datetime' in d) for d in dtype_strs):
        return 'object'
    if any('float' in d for d in dtype_strs):
        return 'float64'
    if all('int' in d for d in dtype_strs):
        return 'int64'
    return 'object'

def load_normalized(f):
    """Read a CSV and normalize column names (uppercase + strip whitespace)."""
    df = pd.read_csv(f, low_memory=False)
    df.columns = df.columns.str.strip().str.upper()
    return df

def apply_stage_1(df, all_columns, col_dtypes):
    """Apply the unified schema to a single DataFrame."""
    for col in all_columns:
        if col not in df.columns:
            df[col] = np.nan
    df = df[all_columns]
    
    for col in all_columns:
        target_t = col_dtypes[col]
        if target_t == 'string':
            df[col] = df[col].astype(str).apply(strip_trailing_float)
            df.loc[df[col].isin(['nan', '<NA>', 'NaN', 'None', '']), col] = pd.NA
        elif target_t == 'object':
            mask = df[col].notna()
            if mask.any():
                df.loc[mask, col] = df.loc[mask, col].astype(str).apply(strip_trailing_float)
            df.loc[df[col].isin(['nan', '<NA>', 'NaN', 'None', '']), col] = pd.NA
        elif target_t == 'float64':
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif target_t == 'int64':
            numeric = pd.to_numeric(df[col], errors='coerce')
            try:
                df[col] = numeric.astype('Int64')
            except (TypeError, ValueError):
                df[col] = numeric
    return df

## Stage 2: Flag Non-Response and Family Shifts

**Why is this stage here?**
Sometimes households skip interviews, or an entirely new family moves into the dwelling but keeps the same `HH_ID`. If we invisibly drop these rows, downstream logic computing "income volatility over time" will be silently corrupted. 

Stage 2 adds explicit `IS_RESPONDING` and `FAMILY_SHIFTED_FLAG` columns. We keep the rows intact so that later stages can consciously decide how to treat them.

In [ ]:
VALID_RESPONSE_VALUES = {"Accepted"}
SHIFTED_VALUES = {"Y"}

def apply_stage_2(df):
    if 'RESPONSE_STATUS' in df.columns:
        df['IS_RESPONDING'] = df['RESPONSE_STATUS'].astype(str).isin(VALID_RESPONSE_VALUES)
    else:
        df['IS_RESPONDING'] = pd.NA

    if 'FAMILY_SHIFTED' in df.columns:
        df['FAMILY_SHIFTED_FLAG'] = df['FAMILY_SHIFTED'].astype(str).isin(SHIFTED_VALUES)
    else:
        df['FAMILY_SHIFTED_FLAG'] = pd.NA
        
    return df

## Stage 3: Resolve Time-Granularity Mismatch

**Why is this stage here?**
CMIE datasets run on different timelines. `household_income` and `consumption_pyramids` are genuinely monthly (each month brings new data). However, `people_of_india` and `Aspirations_of_india` are "wave-based" (recorded once per 4-month wave).

If we leave wave-based data in monthly files, we'll have false duplicates that ruin statistical analysis. 
Stage 3:
- **Monthly Datasets:** Concatenates them into one large panel and attaches a standard `WAVE_LABEL`.
- **Wave Datasets:** Collapses repeated monthly copies into a single row per household (using the *last non-missing value* in that wave).

In [ ]:
MONTHLY_KEEP_DATASETS = ["household_income", "consumption_pyramids"]
WAVE_COLLAPSE_DATASETS = ["people_of_india", "Aspirations_of_india"]

def derive_wave_from_month_num(month_num, year=ANALYSIS_YEAR):
    """Bucket a month into CMIE's Jan-Apr / May-Aug / Sep-Dec waves."""
    if pd.isna(month_num): return pd.NA
    
    m = None
    if isinstance(month_num, str) and not month_num.isdigit():
        month_str = month_num.split()[0][:3].lower()
        month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
                     'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
        m = month_map.get(month_str)
    else:
        try: m = int(month_num)
        except ValueError: pass
            
    if m is None: return pd.NA

    if 1 <= m <= 4: return f"{year}-JanApr"
    elif 5 <= m <= 8: return f"{year}-MayAug"
    elif 9 <= m <= 12: return f"{year}-SepDec"
    return pd.NA

def derive_wave_from_date(date_series):
    """Bucket a date column into Jan-Apr / May-Aug / Sep-Dec of its own year."""
    dt = pd.to_datetime(date_series, errors='coerce')
    def bucket(d):
        if pd.isna(d): return pd.NA
        if 1 <= d.month <= 4: return f"{d.year}-JanApr"
        elif 5 <= d.month <= 8: return f"{d.year}-MayAug"
        else: return f"{d.year}-SepDec"
    return dt.apply(bucket)

def apply_stage_3_monthly(df, ds_name):
    """Simply label the waves and keep all monthly rows."""
    if 'WAVE_NO' in df.columns:
        df['WAVE_LABEL'] = df['WAVE_NO'].astype(str)
    elif 'MONTH' in df.columns:
        df['WAVE_LABEL'] = df['MONTH'].apply(derive_wave_from_month_num)
    return df

def apply_stage_3_wave(df, ds_name):
    """Collapse multiple monthly copies into a single row per wave."""
    if ds_name == "people_of_india":
        group_cols = ['HH_ID', 'MEM_ID', 'WAVE_NO']
        sort_col = 'MONTH_SLOT'
    else:  # Aspirations_of_india
        df['WAVE_NO'] = derive_wave_from_date(df['DATE_OF_INTERVIEW'])
        group_cols = ['HH_ID', 'WAVE_NO']
        sort_col = 'DATE_OF_INTERVIEW'

    # Forward fill missing values then take the last row of the group
    df_sorted = df.sort_values(group_cols + [sort_col])
    filled = df_sorted.groupby(group_cols, sort=False).ffill()
    filled[group_cols] = df_sorted[group_cols]  # ffill drops group cols
    collapsed = filled.groupby(group_cols, as_index=False).tail(1)
    return collapsed

## Stage 4: Deduplicate and Verify Keys

**Why is this stage here?**
Before merging multiple datasets together, we must guarantee that our Primary Keys are unique. If a household has two entries in one wave unexpectedly, performing a SQL-style join later will cause "fan-out" (multiplying rows and ruining totals). Stage 4 audits the unique keys and alerts us to duplicates.

In [ ]:
def check_unique(df, key_cols, label):
    """Verify that key_cols form a unique key in df."""
    key_cols_present = [c for c in key_cols if c in df.columns]
    if not key_cols_present:
        print(f"  [{label}] SKIP - key columns not found")
        return

    n_rows = len(df)
    n_unique = df[key_cols_present].drop_duplicates().shape[0]
    n_dupes = n_rows - n_unique

    if n_dupes == 0:
        print(f"  [{label}] PASS — {n_rows} rows, perfectly unique on {key_cols_present}")
    else:
        print(f"  [{label}] FAIL — {n_dupes} duplicate rows found on {key_cols_present}!")
        dupes_mask = df.duplicated(subset=key_cols_present, keep=False)
        print(df[dupes_mask][key_cols_present].drop_duplicates().head(5).to_string(index=False))

## Stage 5: Clean Variable Coding

**Why is this stage here?**
Survey data contains string flags ('Y'/'N', 'DK', 'Not Applicable') and extreme numeric outliers. If we don't normalize these:
- 'DK' might be treated as 'False', skewing prevalence rates.
- A single data entry error (e.g. 999999999 income) will destroy averages.

This stage recodes flag variables into strict `True`/`False`/`NA` boolean types, sets negative amounts to `NA`, and winsorizes amount columns at the 99th percentile to cap extreme outliers without silently deleting the rows.

In [ ]:
FLAG_PREFIXES = ('HAS_', 'WILL_', 'BOUGHT_', 'IS_')
NO_VALUES  = {'N', 'No', '0', 0}
YES_VALUES = {'Y', 'Yes', '1', 1}
WINSORIZE_PERCENTILE = 99

def recode_flag(series):
    result = pd.array([pd.NA] * len(series), dtype='boolean')
    s = series.astype(str).str.strip()
    result[s.isin({str(v) for v in YES_VALUES})] = True
    result[s.isin({str(v) for v in NO_VALUES})]  = False
    return pd.array(result, dtype='boolean')

def recode_flags_in_df(df):
    flag_cols = [c for c in df.columns if c.startswith(FLAG_PREFIXES) and c != 'IS_RESPONDING']
    for col in flag_cols:
        df[col] = recode_flag(df[col])
    return df

def clean_amounts_in_df(df, group_cols):
    exclude = {'HH_ID', 'MEM_ID', 'WAVE_NO', 'WAVE_LABEL', 'IS_RESPONDING', 'IS_WINSORIZED'}
    amt_cols = [c for c in df.columns if c not in exclude and not c.startswith(FLAG_PREFIXES) 
                and pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]
    
    for col in amt_cols:
        neg_mask = df[col] < 0
        if neg_mask.any():
            df.loc[neg_mask, col] = pd.NA
            
    df['IS_WINSORIZED'] = False
    for col in amt_cols:
        if df[col].isna().all(): continue
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = df[col].astype('float64')
            
        if group_cols and all(g in df.columns for g in group_cols):
            touched_any = pd.Series(False, index=df.index)
            for name, grp_idx in df.groupby(group_cols, dropna=False).groups.items():
                s = df.loc[grp_idx, col]
                if s.dropna().empty: continue
                cap = s.quantile(WINSORIZE_PERCENTILE / 100)
                mask = s > cap
                if mask.any():
                    df.loc[grp_idx[mask], col] = cap
                    touched_any.loc[grp_idx[mask]] = True
            if touched_any.sum() > 0:
                df.loc[touched_any, 'IS_WINSORIZED'] = True
        else:
            cap = df[col].quantile(WINSORIZE_PERCENTILE / 100)
            mask = df[col] > cap
            df.loc[mask, col] = cap
            if mask.sum() > 0: df.loc[mask, 'IS_WINSORIZED'] = True
    return df

def apply_stage_5(df, ds_name):
    if ds_name in ['Aspirations_of_india', 'people_of_india']:
        df = recode_flags_in_df(df)
        
    if ds_name == 'household_income':
        df = clean_amounts_in_df(df, group_cols=['WAVE_LABEL', 'OCCUPATION_GROUP'])
    elif ds_name == 'consumption_pyramids':
        df = clean_amounts_in_df(df, group_cols=['WAVE_LABEL', 'OCCUPATION_GROUP'])
    elif ds_name == 'Aspirations_of_india':
        df = clean_amounts_in_df(df, group_cols=['WAVE_NO', 'OCCUPATION_GROUP'])
    return df

## Stage 6: Collapse People of India to Household Level

**Why is this stage here?**
The `people_of_india` dataset is uniquely measured at the *member* level (e.g. 5 rows for a household of 5 people). Our goal is to merge everything into a unified *household-level* master panel.

Stage 6 extracts demographics for the Head of Household (`_HOH` suffix), applies a strict formal employment rule (`_IS_FORMAL`), and rolls up asset ownership (True if *any* member owns the asset).

In [ ]:
def apply_stage_6(df):
    GROUP_COLS = ['HH_ID', 'WAVE_NO']
    
    # 1. Head of Household (HOH) variables
    HOH_CODE = 'HOH'
    HOH_COLS = [c for c in ['AGE_YRS', 'OCCUPATION', 'NATURE_OF_OCCUPATION', 'TYPE_OF_EMPLOYMENT', 'EMPLOYMENT_STATUS', 'EMPLOYMENT_ARRANGEMENT'] if c in df.columns]
    df_hoh = df[df['RELATION_WITH_HOH'] == HOH_CODE][GROUP_COLS + HOH_COLS].drop_duplicates(subset=GROUP_COLS)
    df_hoh = df_hoh.rename(columns={c: c + '_HOH' for c in HOH_COLS})
    
    # 2. Formality Rule
    FORMAL_TYPES = {'Full-time'}
    INFORMAL_OCC = {'Wage Labourer', 'Agricultural Labourer', 'Small Farmer', 'Organised Farmer', 'Self Employed Entrepreneur', 'Businessman', 'Home-based Worker', 'Small Trader/Hawker/ Businessman without Fixed Premises', 'Industrial Workers', 'Support Staff'}
    if all(c in df.columns for c in ['EMPLOYMENT_STATUS', 'TYPE_OF_EMPLOYMENT', 'NATURE_OF_OCCUPATION']):
        is_emp = df['EMPLOYMENT_STATUS'] == 'Employed'
        is_fulltime = df['TYPE_OF_EMPLOYMENT'].isin(FORMAL_TYPES)
        is_informal = df['NATURE_OF_OCCUPATION'].isin(INFORMAL_OCC)
        df['_IS_FORMAL'] = (is_emp & is_fulltime & ~is_informal).astype('boolean')
        df.loc[df['EMPLOYMENT_STATUS'].isna(), '_IS_FORMAL'] = pd.NA
    else:
        df['_IS_FORMAL'] = pd.Series(pd.NA, index=df.index, dtype='boolean')
        
    # 3. Aggregation (Any-True logic)
    def any_true_agg(s):
        vals = s.dropna()
        return bool(vals.any()) if len(vals) > 0 else pd.NA
        
    agg_dict = {'N_MEMBERS': ('HH_ID', 'count')}
    if 'AGE_YRS' in df.columns:
        df['_IS_ADULT'] = df['AGE_YRS'].ge(18).astype('boolean')
        agg_dict['N_ADULTS'] = ('_IS_ADULT', 'sum')
    if 'EMPLOYMENT_STATUS' in df.columns:
        df['_IS_EMPLOYED'] = (df['EMPLOYMENT_STATUS'] == 'Employed').astype('boolean')
        agg_dict['N_EMPLOYED'] = ('_IS_EMPLOYED', 'sum')
    agg_dict['N_FORMAL'] = ('_IS_FORMAL', 'sum')
    agg_dict['HAS_FORMAL_EMPLOYED_MEM'] = ('_IS_FORMAL', any_true_agg)
    if 'IS_RESPONDING' in df.columns: agg_dict['IS_RESPONDING'] = ('IS_RESPONDING', any_true_agg)
    if 'FAMILY_SHIFTED_FLAG' in df.columns: agg_dict['FAMILY_SHIFTED_FLAG'] = ('FAMILY_SHIFTED_FLAG', any_true_agg)
    
    df_agg = df.groupby(GROUP_COLS, dropna=False).agg(**agg_dict).reset_index()
    
    # 4. Asset Flags
    ASSET_FLAG_COLS = [c for c in df.columns if c.startswith(('HAS_', 'IS_')) and c != 'IS_RESPONDING' and df[c].dtype == 'boolean']
    if ASSET_FLAG_COLS:
        asset_agg = {col: (col, any_true_agg) for col in ASSET_FLAG_COLS}
        df_assets = df.groupby(GROUP_COLS, dropna=False).agg(**asset_agg).reset_index()
        df_agg = df_agg.merge(df_assets, on=GROUP_COLS, how='left')
        
    df_final = df_agg.merge(df_hoh, on=GROUP_COLS, how='left')
    return df_final

## Stage 7: Build Derived Indicators

**Why is this stage here?**
We need complex analytical metrics that span across columns or time. Stage 7 computes:
- `HIGH_COST_DEBT_FLAG` (borrowing from lenders, credit cards, etc.)
- `ILLIQUID_SAVINGS_FLAG` (saving in PF, FD, life insurance)
- `INC_CV` (Income Coefficient of Variation over the 12 months for income volatility)


In [ ]:
def apply_stage_7(asp, inc):
    def any_flag_true(df_subset):
        arr = df_subset.values
        result = pd.array([pd.NA] * len(df_subset), dtype='boolean')
        for i in range(len(df_subset)):
            row = [v for v in arr[i] if not pd.isna(v)]
            if not row: result[i] = pd.NA
            elif any(row): result[i] = True
            else: result[i] = False
        return result
        
    # A. High Cost Debt
    hcd_cols = [c for c in ['BORR_FRM_LENDER', 'BORR_FRM_CC', 'BORR_FRM_NBFC', 'BORR_FRM_MFI', 'BORR_FRM_CHTFUND'] if c in asp.columns]
    if hcd_cols: asp['HIGH_COST_DEBT_FLAG'] = any_flag_true(asp[hcd_cols].astype('boolean'))
    else: asp['HIGH_COST_DEBT_FLAG'] = pd.NA
        
    # B. Illiquid Savings
    ils_cols = [c for c in ['HAS_SAVING_IN_PF', 'HAS_SAVING_IN_LIFE_INS', 'HAS_SAVING_IN_PENSION_SCHEME', 'HAS_SAVING_IN_FD'] if c in asp.columns]
    if ils_cols: asp['ILLIQUID_SAVINGS_FLAG'] = any_flag_true(asp[ils_cols].astype('boolean'))
    else: asp['ILLIQUID_SAVINGS_FLAG'] = pd.NA
        
    # C. Co-Hold Flag
    asp['COHOLD_FLAG'] = asp['HIGH_COST_DEBT_FLAG'].astype('boolean') & asp['ILLIQUID_SAVINGS_FLAG'].astype('boolean')
    
    # D. Income CV
    def compute_cv(s):
        valid = s.dropna()
        if len(valid) < 8 or valid.mean() == 0: return pd.NA
        return valid.std() / valid.mean()
        
    cv_df = inc.groupby('HH_ID')['TOT_INC'].apply(compute_cv).reset_index().rename(columns={'TOT_INC': 'INC_CV'})
    cv_df['INC_CV'] = pd.to_numeric(cv_df['INC_CV'], errors='coerce').astype('Float64')
    
    # Final assembly for stage 7
    core_flags = asp[['HH_ID', 'WAVE_NO', 'HIGH_COST_DEBT_FLAG', 'ILLIQUID_SAVINGS_FLAG', 'COHOLD_FLAG']].copy()
    core_flags = core_flags.merge(cv_df, on='HH_ID', how='left')
    
    return core_flags

## Stage 8: Master Panel Merge

**Why is this stage here?**
This is the grand finale. We have cleaned the wave-based files (`people_of_india`, `Aspirations`) and the monthly files (`household_income`, `consumption_pyramids`), and computed derived indicators.

Because merging 48 large CSVs in pandas can easily exceed memory limits, we write out the intermediate clean files and use **DuckDB** to execute an out-of-core SQL join. DuckDB smartly broadcasts the wave-level data (e.g. demographics) onto the monthly backbone (income/consumption) and streams the final `master_panel.parquet` straight to disk!

In [ ]:
import duckdb

def apply_stage_8(base_dir):
    """
    Uses DuckDB to join the saved Stage 6/7 Parquet files into a Master Panel.
    """
    conn = duckdb.connect()
    
    POI_PATH = os.path.join(base_dir, 'Cleaned_Output', 'people_of_india_hh.parquet')
    ASP_PATH = os.path.join(base_dir, 'Cleaned_Output', 'Aspirations_of_india_final.parquet')
    DER_PATH = os.path.join(base_dir, 'Cleaned_Output', 'stage7_derived_indicators.parquet')
    INC_PATH = os.path.join(base_dir, 'Cleaned_Output', 'household_income_final.parquet')
    CON_PATH = os.path.join(base_dir, 'Cleaned_Output', 'consumption_pyramids_final.parquet')
    
    # 1. Join Wave-Level Data (PoI + Asp + Derived)
    wave_query = f"""
        SELECT 
            COALESCE(p.HH_ID, a.HH_ID) AS HH_ID,
            COALESCE(p.WAVE_NO, a.WAVE_NO) AS WAVE_NO,
            p.* EXCLUDE (HH_ID, WAVE_NO, IS_RESPONDING, FAMILY_SHIFTED_FLAG),
            a.* EXCLUDE (HH_ID, WAVE_NO, IS_RESPONDING, FAMILY_SHIFTED_FLAG),
            p.IS_RESPONDING AS IS_RESPONDING_POI,
            a.IS_RESPONDING AS IS_RESPONDING_ASP,
            d.HIGH_COST_DEBT_FLAG,
            d.ILLIQUID_SAVINGS_FLAG,
            d.COHOLD_FLAG,
            d.INC_CV
        FROM read_parquet('{POI_PATH}') AS p
        FULL OUTER JOIN read_parquet('{ASP_PATH}') AS a ON p.HH_ID = a.HH_ID AND p.WAVE_NO = a.WAVE_NO
        LEFT JOIN read_parquet('{DER_PATH}') AS d ON COALESCE(p.HH_ID, a.HH_ID) = d.HH_ID AND COALESCE(p.WAVE_NO, a.WAVE_NO) = d.WAVE_NO
    """
    conn.execute(f"CREATE VIEW wave_tbl AS {wave_query}")
    
    # 2. Join Monthly Data + Wave Data
    final_query = f"""
        WITH monthly_tbl AS (
            SELECT * FROM read_parquet('{INC_PATH}')
            NATURAL FULL OUTER JOIN read_parquet('{CON_PATH}')
        )
        SELECT m.*, w.* EXCLUDE (HH_ID, WAVE_NO)
        FROM monthly_tbl AS m
        LEFT JOIN wave_tbl AS w ON m.HH_ID = w.HH_ID AND m.WAVE_LABEL = w.WAVE_NO
    """
    conn.execute(f"CREATE VIEW master_tbl AS {final_query}")
    
    out_path = os.path.join(base_dir, 'Cleaned_Output', 'master_panel.parquet')
    print(f"Streaming final master panel to {out_path}...")
    conn.execute(f"COPY (SELECT * FROM master_tbl) TO '{out_path}' (FORMAT PARQUET)")
    print("Master panel successfully created!")
    conn.close()


## Execution Engine

This cell strings all the stages together. It loops over the datasets, processes each month sequentially (to prevent RAM crashes), concatenates them, and outputs the final clean Parquet files.

In [ ]:
out_dir = os.path.join(base_dir, 'Cleaned_Output')
os.makedirs(out_dir, exist_ok=True)

processed_dfs = {}

# STAGES 1 TO 6: Process each dataset individually
for ds in datasets:
    print(f"\n{'='*50}\nProcessing {ds}\n{'='*50}")
    path = os.path.join(base_dir, ds, '*.csv')
    files = sorted(glob.glob(path))
    
    if not files:
        print(f"  WARNING: No CSVs found in {path}")
        continue
        
    # STAGE 1 (Pass 1)
    col_dtype_sightings = {}
    all_columns_seen = set()
    for f in files:
        df_temp = load_normalized(f)
        all_columns_seen.update(df_temp.columns)
        for col in df_temp.columns:
            col_dtype_sightings.setdefault(col, set()).add(str(df_temp[col].dtype))
    
    col_dtypes = {col: ('string' if col in ID_COLS else resolve_dtype(col_dtype_sightings[col])) for col in all_columns_seen}
    all_columns = sorted(all_columns_seen)

    # STAGE 1 (Pass 2) & STAGE 2
    processed_months = []
    for f in files:
        df = load_normalized(f)
        df = apply_stage_1(df, all_columns, col_dtypes)
        df = apply_stage_2(df)
        processed_months.append(df)

    full_df = pd.concat(processed_months, ignore_index=True)
    
    # STAGE 3
    if ds in MONTHLY_KEEP_DATASETS:
        full_df = apply_stage_3_monthly(full_df, ds)
    else:
        full_df = apply_stage_3_wave(full_df, ds)
        
    # STAGE 5
    full_df = apply_stage_5(full_df, ds)
    
    # STAGE 6 (Only for People of India)
    if ds == 'people_of_india':
        full_df = apply_stage_6(full_df)
        check_unique(full_df, ['HH_ID', 'WAVE_NO'], 'People Unique Key Check')
        out_path = os.path.join(out_dir, 'people_of_india_hh.parquet')
    elif ds == 'Aspirations_of_india':
        check_unique(full_df, ['HH_ID', 'WAVE_NO'], 'Aspirations Unique Key Check')
        out_path = os.path.join(out_dir, f'{ds}_final.parquet')
    else:
        check_unique(full_df, ['HH_ID', 'WAVE_LABEL', 'MONTH'], 'Monthly Unique Key Check')
        out_path = os.path.join(out_dir, f'{ds}_final.parquet')
        
    full_df.to_parquet(out_path, index=False)
    print(f"Saved {ds} to {out_path}")
    processed_dfs[ds] = full_df

# STAGE 7: Derived Indicators
if 'Aspirations_of_india' in processed_dfs and 'household_income' in processed_dfs:
    print(f"\n{'='*50}\nProcessing Stage 7: Derived Indicators\n{'='*50}")
    der_df = apply_stage_7(processed_dfs['Aspirations_of_india'], processed_dfs['household_income'])
    der_out = os.path.join(out_dir, 'stage7_derived_indicators.parquet')
    der_df.to_parquet(der_out, index=False)
    print(f"Saved derived indicators to {der_out}")

# STAGE 8: DuckDB Master Panel Merge
print(f"\n{'='*50}\nProcessing Stage 8: DuckDB Master Panel Merge\n{'='*50}")
apply_stage_8(base_dir)
